# Shapelet Source Reconstruction in Gravitational Lensing

## Problem Statement

**Given**: A noisy, blurred, gravitationally lensed image of a distant galaxy, plus a known lens model.

**Goal**: Reconstruct the intrinsic (unlensed) source galaxy from this degraded observation.

This is an **inverse problem** in astrophysical imaging. A massive foreground object (e.g., a galaxy cluster) bends the light from a background source, distorting its image. We must undo this distortion to recover what the source actually looks like.

## Why is this hard?

1. **Ill-posed**: Many different source configurations can produce the same lensed image, especially with noise.
2. **Multiple degradations**: The observation suffers from lensing distortion, PSF blurring, pixelization, and photon/background noise — all at once.
3. **Large dynamic range**: The source galaxy (NGC 1300, a barred spiral) has complex structure spanning many spatial scales.

## Method Overview

### Shapelet Basis Functions
Shapelets are 2D orthonormal basis functions built from Hermite polynomials weighted by a Gaussian:

$$B_{n_1, n_2}(\mathbf{x}; \beta) = \phi_{n_1}(x/\beta) \cdot \phi_{n_2}(y/\beta), \quad \phi_n(u) = \frac{1}{\sqrt{2^n \sqrt{\pi}\, n!}} H_n(u)\, e^{-u^2/2}$$

Any smooth image can be decomposed as $I(\mathbf{x}) = \sum_i c_i B_i(\mathbf{x}; \beta)$. The scale parameter $\beta$ controls the characteristic size of the basis functions.

### Forward Model (Image Simulation)
Given shapelet coefficients $\mathbf{c}$ and lens parameters, simulate the observation:

1. **Ray-trace** image-plane pixels backward through the lens to source-plane coordinates: $\vec{\beta} = \vec{\theta} - \vec{\alpha}(\vec{\theta})$
2. **Evaluate** the shapelet source at the ray-traced positions
3. **Convolve** with the telescope PSF (Gaussian, FWHM = 0.1")
4. **Downsample** from supersampled (5x) to detector resolution
5. **Add noise** (Poisson photon noise + Gaussian background)

### Inverse Problem (Source Reconstruction)
The forward model is **linear** in the shapelet coefficients: $\mathbf{d} = A\mathbf{c} + \mathbf{n}$, where each column of the response matrix $A$ is one basis function propagated through the full imaging pipeline. We solve via **weighted least squares**:

$$\hat{\mathbf{c}} = (A^T W A)^{-1} A^T W \mathbf{d}$$

with noise weights $W_{ii} = 1/\sigma_i^2$ accounting for position-dependent noise (Poisson + background).

A reduced $\chi^2 \approx 1$ indicates the model fits the data to within the noise level.

### This notebook also demonstrates:
- **Image deconvolution** (same framework, no lensing) — recovering a sharp image from a blurry, noisy observation.

**This notebook loads precomputed results** and runs in seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys
%matplotlib inline

# Add task root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Load precomputed reference outputs (runs in seconds)
REF_DIR = os.path.join('..', 'evaluation', 'reference_outputs')
raw = np.load(os.path.join(REF_DIR, 'raw_data.npz'))
lens = np.load(os.path.join(REF_DIR, 'lensing_outputs.npz'))
dc = np.load(os.path.join(REF_DIR, 'deconv_outputs.npz'))

import json
with open(os.path.join(REF_DIR, 'metrics.json')) as f:
    metrics = json.load(f)
print('Metrics:', json.dumps(metrics, indent=2))

## 1. Shapelet Image Decomposition

Before simulating lensing, we need a realistic source galaxy. We decompose an image of **NGC 1300** (a barred spiral galaxy) into shapelet coefficients.

With $n_{\max} = 150$ and $\beta = 10$, we get $(n_{\max}+1)(n_{\max}+2)/2 = 11{,}476$ basis functions. This high-order decomposition captures the galaxy's fine structure (spiral arms, bar, dust lanes).

The preprocessing steps are: background subtraction, padding to square, Gaussian smoothing, and 25x downsampling to make the decomposition computationally tractable.

In [ ]:
f, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, key, title in zip(axes,
    ['ngc_square', 'ngc_conv', 'ngc_resized', 'image_reconstructed'],
    ['original image', 'convolved image', 'resized', 'reconstructed']):
    ax.matshow(raw[key], origin='lower')
    ax.set_title(title)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.autoscale(False)
plt.tight_layout()
plt.show()
print(f"Number of shapelet coefficients: {metrics['num_shapelet_coeffs_decomp']}")

## 2. Gravitational Lensing Simulation

Using the shapelet decomposition as our source galaxy, we simulate the full imaging pipeline. The lens model is a **Singular Isothermal Sphere (SIS)** with Einstein radius $\theta_E = 0.5"$, which deflects light rays by:

$$\vec{\alpha}(\vec{\theta}) = \theta_E \cdot \frac{\vec{\theta}}{|\vec{\theta}|}$$

The source is offset by $(0.2", 0)$ from the optical axis. The five panels below show each stage of degradation: the intrinsic source, the lensed image (note the arc-like distortion), PSF convolution (blurring), pixelization (downsampling from 5x supersampled), and the final noisy observation.

In [ ]:
f, axes = plt.subplots(1, 5, figsize=(20, 4))
keys = ['image_hr_nolens', 'image_hr_lensed', 'image_hr_conv', 'image_no_noise', 'image_real']
labels = ['intrinsic source', '+ lensing effect', '+ convolution', '+ pixelisation', '+ noise']
for ax, key, label in zip(axes, keys, labels):
    ax.matshow(lens[key], origin='lower', extent=[0, 1, 0, 1])
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.autoscale(False)
    ax.text(0.05, 0.05, label, color='w', fontsize=20, transform=ax.transAxes)
f.tight_layout()
f.subplots_adjust(wspace=0., hspace=0.05)
plt.show()

## 3. Source Reconstruction from Lensed Image

Now we solve the inverse problem: given the noisy lensed image (bottom-right of the previous figure) and the known lens model, recover the intrinsic source.

We use a **coarser** shapelet basis ($n_{\max} = 20$, $\beta = 0.15"$, giving 231 coefficients) than the original decomposition (11,476 coefficients). This acts as **implicit regularization** — the limited basis cannot fit noise, only the smooth underlying source structure.

The response matrix $A$ is built by propagating each of the 231 basis functions through the full forward model (lensing + PSF + pixelization). Then we solve the WLS system to get the reconstructed coefficients.

**Top row**: image plane (64x64 detector). **Bottom row**: source plane (320x320 high-resolution).

In [ ]:
f, axes = plt.subplots(2, 3, figsize=(12, 8))
panels = [
    (axes[0,0], lens['image_real'], 'Input image'),
    (axes[0,1], lens['model_lens'], 'Reconstructed image'),
    (axes[0,2], lens['residuals_lens'], 'Image residuals'),
    (axes[1,0], lens['image_hr_nolens'], 'Input source'),
    (axes[1,1], lens['source_recon_2d'], 'Reconstructed source'),
    (axes[1,2], lens['source_recon_2d'] - lens['image_hr_nolens'], 'Source residuals'),
]
for ax, img, title in panels:
    ax.matshow(img, origin='lower')
    ax.set_title(title)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.autoscale(False)
plt.tight_layout()
plt.show()
print(f"Reduced chi2 (lensing): {metrics['chi2_reduced_lensing']:.4f}")
print(f"Source reconstruction residual RMS: {metrics['source_recon_residual_rms']:.4f}")

## 4. Image Deconvolution (No Lensing)

The shapelet + WLS framework generalizes beyond lensing. Here we demonstrate pure **image deconvolution**: given a blurry, noisy image (no gravitational lensing, but a broader PSF with FWHM = 0.25"), recover the sharp source.

The forward model simplifies to: source evaluation + PSF convolution + pixelization + noise. The response matrix $A$ is the same construction but without the ray-tracing step.

This shows the method's versatility — any linear imaging degradation that can be expressed as $\mathbf{d} = A\mathbf{c} + \mathbf{n}$ can be inverted with the same approach.

In [ ]:
f, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, key in zip(axes, ['image_hr_dc', 'image_hr_conv_dc', 'image_no_noise_dc', 'image_real_dc']):
    ax.matshow(dc[key], origin='lower', extent=[0, 1, 0, 1])
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.autoscale(False)
plt.tight_layout()
plt.show()

In [ ]:
f, axes = plt.subplots(2, 3, figsize=(12, 8))
panels = [
    (axes[0,0], dc['image_real_dc'], 'Input image'),
    (axes[0,1], dc['model_dc'], 'Reconstructed image'),
    (axes[0,2], dc['residuals_dc'], 'Image residuals'),
    (axes[1,0], dc['image_hr_dc'], 'Input unconvolved'),
    (axes[1,1], dc['source_deconv_2d'], 'de-convolved image'),
    (axes[1,2], dc['source_deconv_2d'] - dc['image_hr_dc'], 'de-convolution residuals'),
]
for ax, img, title in panels:
    ax.matshow(img, origin='lower')
    ax.set_title(title)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.autoscale(False)
plt.tight_layout()
plt.show()
print(f"Reduced chi2 (deconvolution): {metrics['chi2_reduced_deconv']:.4f}")
print(f"Deconvolution residual RMS: {metrics['deconv_residual_rms']:.4f}")

## 5. Summary

| Aspect | Lensing Reconstruction | Deconvolution |
|--------|----------------------|---------------|
| **Forward model** | Lensing + PSF + pixelization | PSF + pixelization |
| **Lens model** | SIS ($\theta_E = 0.5"$) | None (identity) |
| **PSF FWHM** | 0.1" | 0.25" |
| **Source basis** | 231 shapelets ($n_{\max}=20$) | 231 shapelets ($n_{\max}=20$) |
| **Solver** | Weighted Least Squares | Weighted Least Squares |
| **Reduced $\chi^2$** | ~0.99 | ~1.00 |

**Key insights:**
- **Linearity in coefficients** makes the inverse problem a standard WLS system — no iterative optimization needed
- **Fewer basis functions** ($n_{\max}=20$) than the original decomposition ($n_{\max}=150$) provides natural regularization
- **Supersampling** (5x) is essential for accurate sub-pixel integration of the lensed source light
- **Reduced $\chi^2 \approx 1$** confirms the model explains the data to within the noise level — neither overfitting nor underfitting

To regenerate all outputs from scratch (takes ~1 minute):

```python
# from src.generate_data import generate_all_data
# metrics = generate_all_data(
#     image_path='../../Data/Galaxies/ngc1300.jpg',
#     output_dir='../evaluation/reference_outputs/',
#     seed=42
# )
```

Or run: `cd tasks/shapelet_source_reconstruction && python main.py`